# Basic comparsion of naive and fast convolution

In [167]:
import numpy as np
import sympy as sy
from scipy import signal
from scipy import datasets
from matplotlib import pyplot as plt

In [168]:
from fast_convolution.naive import naive_convolve
from fast_convolution.fast import c3x3_5m20a9e
from fast_convolution.utils import plot_pdf, symmetrical_cyclic_convolution

Lets do a really basic test to compare naive with fast convolution
The input feature will be a simple 5x5 matrix and the output will be a 3x3 matrix

In [169]:
feature = np.arange(8*8).reshape(8, 8)
feature

array([[ 0,  1,  2,  3,  4,  5,  6,  7],
       [ 8,  9, 10, 11, 12, 13, 14, 15],
       [16, 17, 18, 19, 20, 21, 22, 23],
       [24, 25, 26, 27, 28, 29, 30, 31],
       [32, 33, 34, 35, 36, 37, 38, 39],
       [40, 41, 42, 43, 44, 45, 46, 47],
       [48, 49, 50, 51, 52, 53, 54, 55],
       [56, 57, 58, 59, 60, 61, 62, 63]])

In [170]:
weight = np.arange(3*3).reshape(3, 3)
weight

array([[0, 1, 2],
       [3, 4, 5],
       [6, 7, 8]])

In [171]:
np.fliplr(weight)

array([[2, 1, 0],
       [5, 4, 3],
       [8, 7, 6]])

Convolution with no paddin and stride 1
Using convolve2d from scipy, our gold method, it's necessary to reverse the weights to get same result of naive and fast convolution

In [172]:
weight_reversed = weight[::-1, ::-1]
weight_reversed

array([[8, 7, 6],
       [5, 4, 3],
       [2, 1, 0]])

In [173]:
output = signal.convolve2d(feature, weight_reversed, mode='valid')
output

array([[ 474,  510,  546,  582,  618,  654],
       [ 762,  798,  834,  870,  906,  942],
       [1050, 1086, 1122, 1158, 1194, 1230],
       [1338, 1374, 1410, 1446, 1482, 1518],
       [1626, 1662, 1698, 1734, 1770, 1806],
       [1914, 1950, 1986, 2022, 2058, 2094]])

Running naive convolution
9 multiplications and 8 aditions per output scalar

In [174]:
output_naive = naive_convolve(feature, weight)
output_naive

array([[ 474,  510,  546,  582,  618,  654],
       [ 762,  798,  834,  870,  906,  942],
       [1050, 1086, 1122, 1158, 1194, 1230],
       [1338, 1374, 1410, 1446, 1482, 1518],
       [1626, 1662, 1698, 1734, 1770, 1806],
       [1914, 1950, 1986, 2022, 2058, 2094]])

In [175]:
np.all(output == output_naive)

True

Run one fast convolution for each 1d kernel

In [176]:
feature[0]

array([0, 1, 2, 3, 4, 5, 6, 7])

In [177]:
size = 5
step = 3

In [178]:
np.pad(feature[0][r: r+size], (0, size - len(feature[0][r: r+size])), 'constant', constant_values=0)

array([0, 1, 2, 3, 4])

In [179]:
np.pad([0,0,0,0,0], (0, 0), 'constant', constant_values=0)

array([0, 0, 0, 0, 0])

In [180]:
feature[0][0]

0

How join multiple 1d convolution in one 2d convolution

In [181]:
c0 = np.convolve(feature[0], weight[0])
c0

array([ 0,  0,  1,  4,  7, 10, 13, 16, 19, 14])

In [182]:
c1 = np.convolve(feature[1], weight[1])
c1

array([ 24,  59, 106, 118, 130, 142, 154, 166, 130,  75])

In [183]:
c2 = np.convolve(feature[2], weight[2])
c2

array([ 96, 214, 355, 376, 397, 418, 439, 460, 337, 184])

In [184]:
c0+c1+c2

array([120, 273, 462, 498, 534, 570, 606, 642, 486, 273])

In [185]:
fast_conv0 = c3x3_5m20a9e(weight[0].tolist())
out0 = []
r=0
for c in range(0, feature.shape[0], step):
    fast_conv0(np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)).flat()

In [186]:
feature[0][r: r+size] 

array([0, 1, 2, 3, 4])

In [187]:
fast_conv0 = c3x3_5m20a9e(weight[0].tolist())
out0 = [
    # np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)
    fast_conv0(np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)).flat()
    for r in range(0, feature.shape[1]-2)
    for c in range(0, feature.shape[0]-step, step)
]
out0

[[5, 8, 11],
 [14, 17, 20],
 [29, 32, 35],
 [38, 41, 44],
 [53, 56, 59],
 [62, 65, 68],
 [77, 80, 83],
 [86, 89, 92],
 [101, 104, 107],
 [110, 113, 116],
 [125, 128, 131],
 [134, 137, 140]]

In [201]:
np.array(out0).reshape(output.shape)

array([[5, 8, 11, 14, 17, 20],
       [29, 32, 35, 38, 41, 44],
       [53, 56, 59, 62, 65, 68],
       [77, 80, 83, 86, 89, 92],
       [101, 104, 107, 110, 113, 116],
       [125, 128, 131, 134, 137, 140]], dtype=object)

In [188]:
fast_conv1 = c3x3_5m20a9e(weight[1].tolist())
out1 = [
    # feature[r][c: c+size]
    fast_conv1(np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)).flat()
    for r in range(1, feature.shape[1]-1)
    for c in range(0, feature.shape[0]-step, step)
]
out1

[[110, 122, 134],
 [146, 158, 170],
 [206, 218, 230],
 [242, 254, 266],
 [302, 314, 326],
 [338, 350, 362],
 [398, 410, 422],
 [434, 446, 458],
 [494, 506, 518],
 [530, 542, 554],
 [590, 602, 614],
 [626, 638, 650]]

In [189]:
fast_conv2 = c3x3_5m20a9e(weight[2].tolist())
out2 = [
    fast_conv2(np.pad(feature[r][c: c+size], (0, size - len(feature[r][c: c+size])), 'constant', constant_values=0)).flat()
    for r in range(2, feature.shape[1])
    for c in range(0, feature.shape[0]-step, step)
]
out2

[[359, 380, 401],
 [422, 443, 464],
 [527, 548, 569],
 [590, 611, 632],
 [695, 716, 737],
 [758, 779, 800],
 [863, 884, 905],
 [926, 947, 968],
 [1031, 1052, 1073],
 [1094, 1115, 1136],
 [1199, 1220, 1241],
 [1262, 1283, 1304]]

Sum results in the first dimension

In [190]:
output_fast = np.sum([out0, out1, out2], axis=0).reshape(output.shape)
output_fast

array([[474, 510, 546, 582, 618, 654],
       [762, 798, 834, 870, 906, 942],
       [1050, 1086, 1122, 1158, 1194, 1230],
       [1338, 1374, 1410, 1446, 1482, 1518],
       [1626, 1662, 1698, 1734, 1770, 1806],
       [1914, 1950, 1986, 2022, 2058, 2094]], dtype=object)

In [191]:
np.all(output_fast == output_naive)

True

Camparing how much operations are used in naive and fast method

Output Size

In [192]:
size = output.size
size

36

Naive: total of multiplications

In [193]:
size * 9

324

Naive: total of additions

In [194]:
size * 8

288

Fast: total of multiplications

In [195]:
size * 5

180

Fast: additions for each batch processed

In [196]:
add0 = size * 20
add0

720

Fast: additions to join batches

In [197]:
add1 = size * 2
add1

72

Fast: Total of additions

In [198]:
add0 + add1

792

Fast: total of extra operations - bit shifts and etc

In [199]:
size * 9

324